# Realización de Processing con SageMaker



This notebook shows a lightweight example of using SageMaker Processing to create train, test, and validation datasets. SageMaker Processing is used to create these datasets, which then are written back to S3.

## Contents

1. [Prepare resources](#Prepare-resources)
1. [Download data](#Download-data)
1. [Prepare Processing script](#Prepare-Processing-script)
1. [Run Processing job](#Run-Processing-job)
1. [Conclusion](#Conclusion)

## Prepare resources

First, let’s create an SKLearnProcessor object, passing the scikit-learn version we want to use, as well as our managed infrastructure requirements.

Aqui lo que haremos será crear un SKLearnProcessor object, donde indicaremos la versión de scikit-learn que queremos usar y también los requerimientos de la infraestructura

In [ ]:
import boto3 #boto3 es un SDK de Python para interactuar con los servicios de AWS?
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.processing import SKLearnProcessor

sess = sagemaker.Session()
role = get_execution_role()

bucket = sess.default_bucket()
prefix = "sagemaker/1-processing"

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1", role=role, instance_type="ml.m5.xlarge", instance_count=1
)

In [ ]:
print(f"Processing job will persist files in the {bucket} bucket under the {prefix} key")

## Dataset

Usaremos para este lab el dataset del Titanic

## Processing script

Localizado en `processing.py`. Este código lee el dataset completo desde S3; realiza un pequeño feature engineering y divide posteriormente en train, test y validation sets. Después escribe los tres outputs en S3

## Run Processing job

Run the Processing job, specifying the script name, input file, and output files.

Specific paths inside the container can be indicated with the "destination" parameter.

In [ ]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor.run(
    code="processing.py",
    # arguments = ["arg1", "arg2"], # Arguments can optionally be specified here
    inputs=[ProcessingInput(source="dataset.csv", destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(source="/opt/ml/processing/output/train",
                         destination=f"s3://{bucket}/{prefix}/train"),
        ProcessingOutput(source="/opt/ml/processing/output/validation",
                         destination=f"s3://{bucket}/{prefix}/validation"),
        ProcessingOutput(source="/opt/ml/processing/output/test",
                         destination=f"s3://{bucket}/{prefix}/test"),
    ],
)

Confirm that the output dataset files were written to S3.

## Conclusion

In this notebook, we read a dataset from S3 and processed it into train, test, and validation sets using a SageMaker Processing job. You can extend this example for preprocessing your own datasets in preparation for machine learning or other applications.